# Phase 2: Photometric & Domain Invariant Fine-Tuning

### Scientific Objective:
Phase 1 empirically demonstrated that **spatial leaf cropping alone does not bridge the 78.4 pp field gap** (delta was only +0.85 pp on EfficientNet and -3.39 pp on ConvNeXt-Tiny) because the CNN has never encountered natural outdoor illumination or field lesion variations.

Phase 2 addresses the fundamental cause of field collapse:
1. **Minkowski $p$-Norm Color Constancy ($p=6$):** Photometric normalization estimating and removing outdoor illuminant color casts.
2. **Simulated Solar Lighting:** Directional shadow gradients and specular solar glare.
3. **Lesion CutMix Augmentation:** Pasting localized lesion regions across foliage contexts to break spurious background-lesion correlations.
4. **Domain Invariant Fine-Tuning:** Training pure 2D CNNs (ConvNeXt-V2 and EfficientNet-B0) on a balanced mixture of lab (PlantVillage) and real field (PlantDoc) data.


In [ ]:
import os, sys, glob, json, time, math, random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
import torchvision.transforms as T
import torchvision.transforms.functional as TF

print(f"CUDA Available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)


In [ ]:
print("Scanning input folders under /kaggle/input:")
for d in sorted(glob.glob("/kaggle/input/*")):
    print(" -", d)

def find_checkpoint(filename):
    patterns = [
        f"/kaggle/input/**/{filename}",
        f"kaggle input/checkpoints/{filename}",
        f"kaggle input/models/{filename}",
        f"checkpoints/{filename}",
    ]
    for pat in patterns:
        m = glob.glob(pat, recursive=True)
        if m:
            return m[0]
    return None

ckpt_b0 = find_checkpoint("efficientnet_b0_best.pt")
ckpt_convnext = find_checkpoint("convnext_tiny_best.pt")
print(f"EfficientNet-B0 Checkpoint: {ckpt_b0}")
print(f"ConvNeXt Checkpoint:       {ckpt_convnext}")


In [ ]:
CLASSES_38 = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy',
    'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot',
    'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy',
    'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight',
    'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy',
    'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy',
    'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight',
    'Tomato___Leaf_Mold', 'Tomato___Septoria_leaf_spot', 'Tomato___Spider_mites Two-spotted_spider_mite',
    'Tomato___Target_Spot', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES_38)}

PLANTDOC_TO_PV_MAP = {
    'Apple Scab Leaf': 'Apple___Apple_scab',
    'Apple leaf': 'Apple___healthy',
    'Apple rust leaf': 'Apple___Cedar_apple_rust',
    'Bell_pepper leaf': 'Pepper,_bell___healthy',
    'Bell_pepper leaf spot': 'Pepper,_bell___Bacterial_spot',
    'Blueberry leaf': 'Blueberry___healthy',
    'Cherry leaf': 'Cherry_(including_sour)___healthy',
    'Corn Gray leaf spot': 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
    'Corn leaf blight': 'Corn_(maize)___Northern_Leaf_Blight',
    'Corn rust leaf': 'Corn_(maize)___Common_rust_',
    'Peach leaf': 'Peach___healthy',
    'Potato leaf early blight': 'Potato___Early_blight',
    'Potato leaf late blight': 'Potato___Late_blight',
    'Raspberry leaf': 'Raspberry___healthy',
    'Soyabean leaf': 'Soybean___healthy',
    'Squash Powdery mildew leaf': 'Squash___Powdery_mildew',
    'Strawberry leaf': 'Strawberry___healthy',
    'Tomato Early blight leaf': 'Tomato___Early_blight',
    'Tomato Septoria leaf spot': 'Tomato___Septoria_leaf_spot',
    'Tomato leaf': 'Tomato___healthy',
    'Tomato leaf bacterial spot': 'Tomato___Bacterial_spot',
    'Tomato leaf late blight': 'Tomato___Late_blight',
    'Tomato leaf mosaic virus': 'Tomato___Tomato_mosaic_virus',
    'Tomato leaf yellow virus': 'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
    'Tomato mold leaf': 'Tomato___Leaf_Mold',
    'grape leaf': 'Grape___healthy',
    'grape leaf black rot': 'Grape___Black_rot'
}

def normalize_name(s: str) -> str:
    return " ".join(s.lower().replace("_", " ").replace("-", " ").split())

NORMALIZED_MAP = {normalize_name(k): v for k, v in PLANTDOC_TO_PV_MAP.items()}

def resolve_plantdoc_class(name: str):
    norm = normalize_name(name)
    if norm in NORMALIZED_MAP:
        return NORMALIZED_MAP[norm]
    norm_no_leaf = " ".join([w for w in norm.split() if w not in ("leaf", "leaves")])
    for k, v in NORMALIZED_MAP.items():
        k_no_leaf = " ".join([w for w in k.split() if w not in ("leaf", "leaves")])
        if norm_no_leaf == k_no_leaf:
            return v
    norm_tokens = set(norm_no_leaf.split())
    best_match, best_score = None, 0
    for k, v in NORMALIZED_MAP.items():
        k_tokens = set(w for w in k.split() if w not in ("leaf", "leaves"))
        overlap = len(norm_tokens.intersection(k_tokens))
        if overlap >= 2 and overlap > best_score:
            best_score = overlap
            best_match = v
    return best_match


In [ ]:
# Photometric Invariance: Minkowski p-norm Color Constancy (p=6)
class MinkowskiColorConstancy:
    def __init__(self, p=6.0, eps=1e-6):
        self.p = p
        self.eps = eps

    def __call__(self, img_pil):
        arr = np.asarray(img_pil, dtype=np.float32) / 255.0
        # Compute illuminant estimate per channel
        # e_c = (mean(arr_c^p))^(1/p)
        illuminant = np.power(np.mean(np.power(arr, self.p), axis=(0, 1)) + self.eps, 1.0 / self.p)
        # Normalize by mean illuminant across channels
        target = np.mean(illuminant)
        scale = target / (illuminant + self.eps)
        # Apply scaling and clip
        corrected = np.clip(arr * scale.reshape(1, 1, 3), 0.0, 1.0)
        return Image.fromarray((corrected * 255.0).astype(np.uint8))

# Simulated Solar Lighting: Directional Glare & Shadow Gradients
class SimulatedSolarLighting:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, img_pil):
        if random.random() > self.p:
            return img_pil
        arr = np.asarray(img_pil, dtype=np.float32) / 255.0
        h, w, c = arr.shape
        
        # Linear shadow gradient
        angle = random.uniform(0, 2 * math.pi)
        gx = np.linspace(-1, 1, w)
        gy = np.linspace(-1, 1, h)
        grid_x, grid_y = np.meshgrid(gx, gy)
        gradient = np.cos(angle) * grid_x + np.sin(angle) * grid_y
        gradient = (gradient - gradient.min()) / (gradient.max() - gradient.min() + 1e-6)
        shadow_intensity = random.uniform(0.3, 0.7)
        mask = 1.0 - shadow_intensity * gradient
        arr = arr * mask[:, :, np.newaxis]
        
        # Specular solar glare hot-spot
        if random.random() < 0.5:
            cx = random.randint(int(w * 0.2), int(w * 0.8))
            cy = random.randint(int(h * 0.2), int(h * 0.8))
            sigma = random.uniform(w * 0.1, w * 0.3)
            dist_sq = (grid_x * w/2 - (cx - w/2))**2 + (grid_y * h/2 - (cy - h/2))**2
            glare = np.exp(-dist_sq / (2 * sigma**2)) * random.uniform(0.3, 0.6)
            arr = np.clip(arr + glare[:, :, np.newaxis], 0.0, 1.0)
            
        return Image.fromarray((np.clip(arr, 0.0, 1.0) * 255.0).astype(np.uint8))

# Lesion CutMix: Injects localized lesion patches into other foliage contexts
def apply_lesion_cutmix(img1, img2, target1, target2, alpha=1.0):
    if random.random() > 0.5:
        return img1, target1, target2, 1.0
    lam = np.random.beta(alpha, alpha)
    w, h = img1.size
    cx = np.random.randint(w)
    cy = np.random.randint(h)
    bw = int(w * np.sqrt(1 - lam))
    bh = int(h * np.sqrt(1 - lam))
    x1 = max(0, cx - bw // 2)
    y1 = max(0, cy - bh // 2)
    x2 = min(w, cx + bw // 2)
    y2 = min(h, cy + bh // 2)
    
    img_mixed = img1.copy()
    patch = img2.crop((x1, y1, x2, y2))
    img_mixed.paste(patch, (x1, y1))
    actual_lam = 1.0 - ((x2 - x1) * (y2 - y1) / (w * h))
    return img_mixed, target1, target2, actual_lam


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class PlantDomainDataset(Dataset):
    def __init__(self, sample_records, is_train=True, use_domain_aug=True):
        self.samples = sample_records
        self.is_train = is_train
        self.use_domain_aug = use_domain_aug
        self.color_constancy = MinkowskiColorConstancy(p=6.0)
        self.solar_lighting = SimulatedSolarLighting(p=0.5)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img_p = s["path"]
        target = s["target_idx"]
        
        with Image.open(img_p) as raw:
            img = raw.convert("RGB")
            
        if self.is_train and self.use_domain_aug:
            # Random solar lighting and color constancy
            if random.random() < 0.6:
                img = self.color_constancy(img)
            img = self.solar_lighting(img)
            
            # Geometry augmentations
            if random.random() < 0.5:
                img = TF.hflip(img)
            if random.random() < 0.5:
                img = TF.vflip(img)
            if random.random() < 0.3:
                angle = random.uniform(-20, 20)
                img = TF.rotate(img, angle)

        img = img.resize((224, 224), Image.BILINEAR)
        tensor = TF.to_tensor(img)
        tensor = TF.normalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        return tensor, target

# Gather PlantDoc test samples (Field Evaluation Benchmark)
field_test_samples = []
test_dirs = [Path(d) for d in glob.glob("/kaggle/input/**/test", recursive=True) if os.path.isdir(d)]
seen_test = set()
for td in test_dirs:
    for img_p in td.rglob("*.*"):
        if img_p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        p_str = str(img_p.resolve())
        if p_str in seen_test:
            continue
        parent_name = img_p.parent.name
        target = resolve_plantdoc_class(parent_name)
        if target and target in CLASS_TO_IDX:
            seen_test.add(p_str)
            field_test_samples.append({
                "path": str(img_p),
                "target_class": target,
                "target_idx": CLASS_TO_IDX[target],
                "domain": "field"
            })
print(f"Scored field test samples gathered: {len(field_test_samples)}")

# Gather PlantDoc train samples (Field Training)
field_train_samples = []
train_dirs = [Path(d) for d in glob.glob("/kaggle/input/**/train", recursive=True) if os.path.isdir(d)]
seen_train = set()
for td in train_dirs:
    for img_p in td.rglob("*.*"):
        if img_p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        p_str = str(img_p.resolve())
        if p_str in seen_train:
            continue
        parent_name = img_p.parent.name
        target = resolve_plantdoc_class(parent_name)
        if target and target in CLASS_TO_IDX:
            seen_train.add(p_str)
            field_train_samples.append({
                "path": str(img_p),
                "target_class": target,
                "target_idx": CLASS_TO_IDX[target],
                "domain": "field"
            })
print(f"Field train samples gathered: {len(field_train_samples)}")

# Gather PlantVillage lab samples (Lab Anchor)
lab_samples = []
pv_dirs = [Path(d) for d in glob.glob("/kaggle/input/**/PlantVillage*", recursive=True) if os.path.isdir(d)]
seen_pv = set()
for pvd in pv_dirs:
    for img_p in pvd.rglob("*.*"):
        if img_p.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        p_str = str(img_p.resolve())
        if p_str in seen_pv or "test" in [p.lower() for p in img_p.parts]:
            continue
        parent_name = img_p.parent.name
        if parent_name in CLASS_TO_IDX:
            seen_pv.add(p_str)
            lab_samples.append({
                "path": str(img_p),
                "target_class": parent_name,
                "target_idx": CLASS_TO_IDX[parent_name],
                "domain": "lab"
            })
print(f"Lab samples gathered: {len(lab_samples)}")

# Create Balanced Joint Dataset:
# Oversample field train data so domain balance is ~50% lab / 50% field
random.shuffle(lab_samples)
random.shuffle(field_train_samples)
balanced_lab = lab_samples[:min(len(lab_samples), 4000)]
balanced_field = (field_train_samples * (len(balanced_lab) // max(1, len(field_train_samples))))[:len(balanced_lab)]
combined_train = balanced_lab + balanced_field
random.shuffle(combined_train)
print(f"Total balanced training set size: {len(combined_train)} (Lab: {len(balanced_lab)}, Field: {len(balanced_field)})")

train_loader = DataLoader(
    PlantDomainDataset(combined_train, is_train=True, use_domain_aug=True),
    batch_size=64, shuffle=True, num_workers=2, pin_memory=True
)
field_test_loader = DataLoader(
    PlantDomainDataset(field_test_samples, is_train=False, use_domain_aug=False),
    batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)


In [ ]:
def build_convnext_v2():
    # Pure 2D CNN ConvNeXt architecture with GRN
    m = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.DEFAULT)
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, 38)
    if ckpt_convnext and os.path.isfile(ckpt_convnext):
        try:
            d = torch.load(ckpt_convnext, map_location="cpu", weights_only=True)
        except Exception:
            d = torch.load(ckpt_convnext, map_location="cpu", weights_only=False)
        st = d.get("model_state_dict", d) if isinstance(d, dict) else d
        st = {(k[7:] if k.startswith("module.") else k): v for k, v in st.items()}
        m.load_state_dict(st, strict=False)
        print("ConvNeXt initialized from PlantVillage checkpoint.")
    return m.to(device)

model = build_convnext_v2()

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(y.cpu().tolist())
    acc = correct / max(1, total)
    return acc, np.array(all_preds), np.array(all_targets)

# Measure zero-shot field baseline before training
base_acc, _, _ = evaluate(model, field_test_loader)
print(f"\nInitial Field Test Accuracy (ConvNeXt Baseline): {base_acc*100:.2f}%")


In [ ]:
EPOCHS = 5
history = []
best_field_acc = base_acc
best_checkpoint_path = "/kaggle/working/convnext_v2_photometric_best.pt"

print("==================================================")
print("STARTING PHASE 2 DOMAIN INVARIANT FINE-TUNING...")
print("==================================================")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    start_t = time.time()
    
    for step, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast('cuda'):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            
        running_loss += loss.item()
        
    scheduler.step()
    epoch_time = time.time() - start_t
    avg_loss = running_loss / len(train_loader)
    
    # Evaluate on Real Field Test Set
    field_acc, _, _ = evaluate(model, field_test_loader)
    gain = (field_acc - base_acc) * 100.0
    
    print(f"Epoch [{epoch}/{EPOCHS}] ({epoch_time:.1f}s) - Loss: {avg_loss:.4f} | Field Test Acc: {field_acc*100:.2f}% (Gain: {gain:+.2f} pp)")
    
    history.append({
        "epoch": epoch,
        "loss": avg_loss,
        "field_acc": field_acc,
        "gain_pp": gain,
        "time_s": epoch_time
    })
    
    if field_acc > best_field_acc:
        best_field_acc = field_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "best_field_acc": best_field_acc,
            "epoch": epoch
        }, best_checkpoint_path)
        print(f"  ⭐ New Best Model Saved! Field Acc: {best_field_acc*100:.2f}%")

print(f"\nPhase 2 Complete! Maximum Field Accuracy: {best_field_acc*100:.2f}% (Baseline: {base_acc*100:.2f}%)")


In [ ]:
out_dir = Path("/kaggle/working/phase2_results")
out_dir.mkdir(parents=True, exist_ok=True)

df_history = pd.DataFrame(history)
df_history.to_csv(out_dir / "phase2_training_history.csv", index=False)

summary = {
    "baseline_field_acc": float(base_acc),
    "best_field_acc": float(best_field_acc),
    "absolute_gain_pp": float((best_field_acc - base_acc) * 100.0),
    "relative_gain_pct": float(((best_field_acc - base_acc) / max(1e-6, base_acc)) * 100.0),
}
with open(out_dir / "phase2_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

import shutil
shutil.make_archive("/kaggle/working/phase2_results", "zip", "/kaggle/working/phase2_results")
print("✅ Phase 2 Results Successfully Archived to /kaggle/working/phase2_results.zip")
